PROCESAMIENTO DE ARCHIVOS .TIF

In [23]:
!pip install rasterio

import os
import glob
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/descargas_inegi'
archivos_encontrados = glob.glob(os.path.join(base_path, '*.tif'))

# Definimos el CRS de destino (UTM Zona 14N - ITRF2008)
dst_crs = 'EPSG:6369' # ITRF2008 / UTM zone 14N

def reproject_to_v14(file_path):
    """Reproyecta un archivo a la Zona 14N si es necesario."""
    with rasterio.open(file_path) as src:
        if src.crs == dst_crs:
            return rasterio.open(file_path)

        print(f"Reproyectando {os.path.basename(file_path)} a Zona 14N...")
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        # Creamos un archivo temporal en memoria para no saturar el Drive
        from rasterio.io import MemoryFile
        memfile = MemoryFile()
        dst = memfile.open(**kwargs)

        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear)
        return dst

if not archivos_encontrados:
    print("No se encontraron archivos.")
else:
    src_files_to_mosaic = []
    try:
        # Procesar cada archivo asegurando el mismo CRS
        for fp in archivos_encontrados:
            temp_src = reproject_to_v14(fp)
            src_files_to_mosaic.append(temp_src)

        print("Iniciando fusión de mosaico...")
        mosaic, out_trans = merge(src_files_to_mosaic)

        # Configurar salida
        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
            "crs": dst_crs
        })

        output_filename = os.path.join(base_path, "CEM_Mosaico_Corregido.tif")
        with rasterio.open(output_filename, "w", **out_meta) as dest:
            dest.write(mosaic)

        print(f"¡Éxito! Mosaico guardado en: {output_filename}")

    except Exception as e:
        print(f"Ocurrió un error: {e}")
    finally:
        for src in src_files_to_mosaic:
            src.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reproyectando e14b49b2_mt.tif a Zona 14N...
Reproyectando e14b49b1_mt.tif a Zona 14N...
Reproyectando e14b49b3_mt.tif a Zona 14N...
Reproyectando e14b49c3_mt.tif a Zona 14N...
Reproyectando e14b49b4_mt.tif a Zona 14N...
Reproyectando e14b49e1_mt.tif a Zona 14N...
Reproyectando e14b49e2_mt.tif a Zona 14N...
Reproyectando e14b49e3_mt.tif a Zona 14N...
Reproyectando e14b49f1_mt.tif a Zona 14N...
Reproyectando e14b49e4_mt.tif a Zona 14N...
Reproyectando e14b49f3_mt.tif a Zona 14N...
Reproyectando e14b49f4_mt.tif a Zona 14N...
Reproyectando e15a41d3_mt.tif a Zona 14N...
Iniciando fusión de mosaico...


/usr/local/lib/python3.12/dist-packages/rasterio/merge.py:369: UserWarning: Ignoring nodata value. The nodata value, -3.4028234663852886e+38, cannot safely be represented in the chosen data type, float32. Consider overriding it using the --nodata option for better results. Falling back to first source's nodata value.
  warnings.warn(


¡Éxito! Mosaico guardado en: /content/drive/MyDrive/descargas_inegi/CEM_Mosaico_Corregido.tif
